In [102]:
!pip install pyserial

In [103]:
import serial, time
!pip install pyserial

In [104]:
#ser.close()

**Note:** if importing `serial` causes an error, you need to install the `pyserial` module using `pip`:

`pip install pyserial`

or 

`pip3 install pyserial`

Windows users should use the Anaconda prompt.  Mac users should be able to use the terminal.

In [105]:
print(serial)

<module 'serial' from 'C:\\Users\\caleb\\anaconda3\\Lib\\site-packages\\serial\\__init__.py'>


In [106]:
print(serial.__file__)

C:\Users\caleb\anaconda3\Lib\site-packages\serial\__init__.py


In [107]:
print(serial.__version__)

3.5


In [108]:
serial.VERSION

'3.5'

**Note:** if you serial version is 2.x, we might need to make changes to the code below

In [109]:
baudrate = 115200

In [110]:
#portname = '/dev/cu.usbmodem11301'#mac
portname = 'COM4'#windows

In [111]:
#ser.close()

In [112]:
ser = serial.Serial(portname, baudrate, timeout=5)

In [113]:
ser.in_waiting

0

In [114]:
#ser.close()

In [115]:
#read_all(ser)

In [116]:
def read_all(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [117]:
read_all(ser)

'dual servo control over serial\n'

In [118]:
def read_one_line(ser):
    out = []
    while ser.in_waiting > 0:
        data1b = ser.read(1)
        data1 = data1b.decode('utf-8')
        if data1 in ['\n','\r']:
            break
        out.append(data1)
        
    outstr = ''.join(out)
    return outstr

In [119]:
read_one_line(ser)

''

In [120]:
read_all(ser)

''

In [121]:
def one_byte_int_to_serial_byte(int_byte):
    out_byte = int(int_byte).to_bytes(1, byteorder='big')
    return out_byte

In [122]:
def WriteByte(ser, bytein):
    out_byte = one_byte_int_to_serial_byte(bytein)
    ser.write(out_byte)

## Example

In [123]:
#byte1 = 7
#WriteByte(ser,byte1)#<--
#time.sleep(0.1)
#byte2 = 156
#WriteByte(ser,byte2)#<--
#time.sleep(0.1)
#next_line = read_one_line(ser)
#extra = read_all(ser)
#print('next_line: %s' % next_line)
#print('extra: %s' % extra)

# Break an integer into two bytes

In [124]:
%matplotlib inline
import matplotlib.pyplot as plt
import numpy as np
from numpy import sin, cos, tan, pi
import robotics
from robotics import Rx, Ry, Rz, sind, cosd, DH, prettymat
rtd = 180/pi
dtr = pi/180

In [147]:
# inputs from user
xll =15 # x origin
yll = 32 # y origin
w =  3.75  # width
h = 5.25   # height
#N = 2
K = 1  # number of steps per side
N = (2*K) #refining the robots movement

In [148]:
#define step size
dx = w/N
dy = h/N

#generate bottom coordinants
x_bottom = np.linspace(xll, xll+w-dx, N)  
y_bottom = np.full(N, yll)

#generate right coordinants
x_right = np.full(N, xll + w)  
y_right = np.linspace(yll, yll+h-dy, N)

#generate top coordinants
x_top = np.linspace(xll+w, xll+dx, N)  
y_top = np.full(N,yll+h)

#generate left coordinants
x_left = np.full(N+2, xll)  
y_left = np.linspace(yll+h, yll, N+1)
y_left = np.append(y_left, yll)
#print(x_left)
#print(y_left)
#combine bottom, right, top, left into one array
x_path = np.concatenate((x_bottom, x_right, x_top, x_left), axis=0) 
y_path = np.concatenate((y_bottom, y_right, y_top, y_left), axis=0) 

#combine x and y into one array
tip_path = np.column_stack((x_path, y_path))
tip_path

array([[15.   , 32.   ],
       [16.875, 32.   ],
       [18.75 , 32.   ],
       [18.75 , 34.625],
       [18.75 , 37.25 ],
       [16.875, 37.25 ],
       [15.   , 37.25 ],
       [15.   , 34.625],
       [15.   , 32.   ],
       [15.   , 32.   ]])

In [149]:
#Define Link lengths
l1 = 23.6# base link
l2 = 24.5 # tip link

#distance from origin to tip
r_squared = tip_path[:,0]**2 + tip_path[:,1]**2

#law of cos for angle between links
alpha_temp = (r_squared-l1**2-l2**2)/(-2*l1*l2)
print(alpha_temp)
alpha = np.arccos(alpha_temp)

print(alpha*rtd)

#vertical angle theorem for theta 2
theta2 = 180 - alpha*rtd

#triangle in link1 co-ordinant system for psi
psi = np.arctan2(l2*sind(theta2),l1+l2*cosd(theta2))*rtd

#angle of r to x-axis
beta = np.arctan2(tip_path[:,1],tip_path[:,0])*rtd

#difference in beta and psi is theta 1
theta1 = beta - psi

##theta2 = 180 - theta2

# check that values are possible before continuing
Check = 0
for i in range(len(theta1)):
    
    if theta2[i] < 0:  #Makes sure theta 1 is positive so positive values of 1000-2000 can be sent to the arduino
        theta2[i] = theta2[i]+360
for i in range(len(theta1)):
    if theta1[i] < 0 or theta1[i] >180: #The servo cannot reach lower than 0 or higher than 180
        Check = 1
    if (theta2[i] > 90 and theta2[i] < 270) or (theta2[i] < 0): #The servo has 1000 -> theta2=90, 2000 -> theta2 = -90 checks to make sure values are not out of range
        Check = 1
if Check == 0:
    print("Correct")
if Check == 1:
    print("Wrong")
print("\ntheta 1:\n",theta1)
print("theta 2:\n",theta2)

[-0.07937565 -0.13105813 -0.18882091 -0.34005805 -0.50321256 -0.44544978
 -0.3937673  -0.23061279 -0.07937565 -0.07937565]
[ 94.55267891  97.53074204 100.88398184 109.88041062 120.21276959
 116.45211912 113.18911557 103.33315166  94.55267891  94.55267891]
Correct

theta 1:
 [21.17151239 20.02114768 19.18885989 25.75164901 32.7714455  33.19057374
 33.95366761 27.3960701  21.17151239 21.17151239]
theta 2:
 [85.44732109 82.46925796 79.11601816 70.11958938 59.78723041 63.54788088
 66.81088443 76.66684834 85.44732109 85.44732109]


In [150]:
#convert angle into arduino code
theta_min = 0     # minimum angle
theta_max = 180   # maximum angle
min_new = 1000    # minimum servo value
max_new = 2000    # maximum servo value

#linear interpolate for the first theta value
myint = min_new + ((theta1-theta_min)*(max_new-min_new))/(theta_max-theta_min)
print(myint)

#linear interpolate for the second theta value; 180 and 90 included to account for robots home position as The servo has 1000 -> theta2=90, 2000 -> theta2 = -90
myint2 = min_new + (((180-(90+theta2))-theta_min)*(max_new-min_new))/(theta_max-theta_min)
print('\n',myint2)


[1117.61951327 1111.22859822 1106.60477719 1143.0647167  1182.0635861
 1184.39207632 1188.6314867  1152.20038944 1117.61951327 1117.61951327]

 [1025.2926606  1041.83745579 1060.46656576 1110.44672565 1167.84871992
 1146.95621732 1128.82841981 1074.07306476 1025.2926606  1025.2926606 ]


In [151]:
def break_into_two(breakint):
    MSB = breakint // 256
    LSB = breakint % 256
    return MSB, LSB

In [152]:
#define arrays to hold the four bytes
byte1 = np.zeros(len(theta1), dtype=int)
byte2 = np.zeros(len(theta1), dtype=int)
byte3 = np.zeros(len(theta1), dtype=int)
byte4 = np.zeros(len(theta1), dtype=int)

#store bytes into temp values to be sent to arduino
for i in range(len(theta1)):
    byte1[i], byte2[i] = break_into_two(myint[i])
    byte3[i], byte4[i] = break_into_two(myint2[i])

print(byte1,'\n\n',byte2,'\n\n\n',byte3,'\n\n',byte4)

[4 4 4 4 4 4 4 4 4 4] 

 [ 93  87  82 119 158 160 164 128  93  93] 


 [4 4 4 4 4 4 4 4 4 4] 

 [  1  17  36  86 143 122 104  50   1   1]


In [153]:
# Send all path points to both servos
# In order to have intermediate points, the loop will iterate twice, and then wait for the arduino to send those bytes to the servo
# The servo has a delay after the second movement but not after the intermediate movement.

x = 1
for i in range(len(theta1)):
    WriteByte(ser, int(byte1[i]))   # servo 1 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte2[i]))   # servo 1 LSB
    time.sleep(0.05)
    WriteByte(ser, int(byte3[i]))   # servo 2 MSB
    time.sleep(0.05)
    WriteByte(ser, int(byte4[i]))   # servo 2 LSB
    time.sleep(0.05)

    # read back confirmation from Arduino
    #line1 = read_one_line(ser)  # servo 1 bytes echo
   # line2 = read_one_line(ser)  # servo 1 int echo
    #line3 = read_one_line(ser)  # servo 2 bytes echo
    #line4 = read_one_line(ser)  # servo 2 int echo
    #print(f"Step {i}: servo1={line2}  servo2={line4}")

   # print('\ntheta1:',theta1[i],'\ntheta2:',theta2[i],'\n','\n\n')
 
    if x == 1:
        x = 2
        
    else:
        
        while 1==1:
            response = ser.readline().decode('utf-8').strip()
            print(response)
            if response == "Ready":
                break
            else: 
                line1 = read_one_line(ser)  # servo 1 bytes echo
                line2 = read_one_line(ser)  # servo 1 int echo
                print(f"Step {i-1}: servo1={line1}  servo2={line2}")
                line3 = read_one_line(ser)  # servo 1 bytes echo
                line4 = read_one_line(ser)  # servo 1 int echo
                                              # servo 2 int echo
                print(f"Step {i}: servo1={line3}  servo2={line4}") 
            #response = ser.readline().decode('utf-8').strip()
            
                
        x = 1
    #time.sleep(1)  # pause between steps so servo has time to move


Step 0: servo1=My int: 1117  servo2=My int2: 1025
Step 1: servo1=My int3: 1111  servo2=My int4: 1041
Ready

Step 2: servo1=My int: 1106  servo2=My int2: 1060
Step 3: servo1=My int3: 1143  servo2=My int4: 1110
Ready

Step 4: servo1=My int: 1182  servo2=My int2: 1167
Step 5: servo1=My int3: 1184  servo2=My int4: 1146
Ready

Step 6: servo1=My int: 1188  servo2=My int2: 1128
Step 7: servo1=My int3: 1152  servo2=My int4: 1074
Ready

Step 8: servo1=My int: 1117  servo2=My int2: 1025
Step 9: servo1=My int3: 1117  servo2=My int4: 1025
Ready


In [144]:
#byte3, byte4 = break_into_two(1200)

In [145]:
#WriteByte(ser, MSB)
#WriteByte(ser, LSB)

In [146]:

WriteByte(ser,byte1)#<--
time.sleep(0.1)

WriteByte(ser,byte2)#<--
time.sleep(0.1)

#WriteByte(ser,byte3)#<--
#time.sleep(0.1)

#WriteByte(ser,byte4)#<--
#time.sleep(0.1)
next_line = read_one_line(ser)
extra = read_all(ser)
print('next_line: %s' % next_line)
print('extra: %s' % extra)

TypeError: only length-1 arrays can be converted to Python scalars

In [ ]:
byte3, byte4 = break_into_two(1500)

In [ ]:
WriteByte(ser,byte3)#<--
time.sleep(0.1)

WriteByte(ser,byte4)#<--
time.sleep(0.1)
#WriteByte(ser,byte3)#<--
#time.sleep(0.1)

#WriteByte(ser,byte4)#<--
#time.sleep(0.1)
next_line = read_one_line(ser)
extra = read_all(ser)
print('next_line: %s' % next_line)
print('extra: %s' % extra)

In [286]:
#ser.close()

In [ ]:
for i in range(1100, 1900, 50):
    byte3, byte4 = break_into_two(i)
    WriteByte(ser,byte3)#<--
    time.sleep(0.1)

    WriteByte(ser,byte4)#<--
    time.sleep(0.1)
    #WriteByte(ser,byte3)#<--
    #time.sleep(0.1)

    #WriteByte(ser,byte4)#<--
    #time.sleep(0.1)
    next_line = read_one_line(ser)
    extra = read_all(ser)
    print('next_line: %s' % next_line)
    print('extra: %s' % extra)
    time.sleep(0.5)
    print(i) 

- How do we break this into two bytes?
- How do we find the most significant byte?
- How do we find the least significant byte?

In [ ]:
ser.close()